# Sentiment Analysis of Flipkart Product Reviews

**Research internship, IISc Bangalore (RBCCPS) x strategicERP, May–Jul 2024**

Predicts whether a customer is satisfied (rating 4–5) or not (rating 1–3) from the
text of their review, using Word2Vec embeddings and a Bidirectional LSTM. The trained
model is served through a Streamlit front end (`app.py`).

**Dataset:** Flipkart Product Reviews (Kaggle) — ADD DATASET LINK — download
`flipkart_product.csv` and place it next to this notebook (or update `DATA_PATH`).

## 1. Setup

In [ ]:
# On current Colab / TensorFlow 2.16+, use legacy Keras 2 APIs (Tokenizer, pad_sequences)
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import re
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
nltk.download("stopwords")
nltk.download("wordnet")
from nltk.corpus import stopwords

from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             classification_report, confusion_matrix)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dropout, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

STOPWORDS = set(stopwords.words("english"))

## 2. Load and clean the data

In [ ]:
DATA_PATH = "flipkart_product.csv"   # in Colab with Drive: "/content/drive/MyDrive/flipkart_product.csv"

raw = pd.read_csv(DATA_PATH, encoding="latin-1")
df = raw.drop(columns=["ProductName", "Price"]).dropna().copy()

# Keep only rows where the rating is a valid integer
def is_integer(value):
    try:
        int(value)
        return True
    except ValueError:
        return False

df = df[df["Rate"].apply(is_integer)].copy()
df["Rate"] = df["Rate"].astype(int)
print(f"Rows after cleaning: {len(df):,}")

## 3. Text preprocessing
Stopword removal → lemmatization → lowercasing → remove non-alphabetic characters.
Lemmatization is preferred over stemming because it returns real root words
(e.g. *easily* → *easy* rather than *easi*).

In [ ]:
_ws_tokenizer = nltk.tokenize.WhitespaceTokenizer()
_lemmatizer = nltk.stem.WordNetLemmatizer()

def preprocess(text: str) -> str:
    words = [w for w in str(text).split() if w not in STOPWORDS]
    words = [_lemmatizer.lemmatize(w) for w in _ws_tokenizer.tokenize(" ".join(words))]
    text = " ".join(words).lower()
    return re.sub("[^a-zA-Z]", " ", text)

df["clean"] = df["Summary"].apply(preprocess)
df[["Summary", "clean", "Rate"]].head()

## 4. Labels: rating 4–5 = positive (1), 1–3 = negative (0)

In [ ]:
labels = (df["Rate"] > 3).astype(int).values

pos_share = labels.mean()
print(f"Positive: {pos_share:.1%}  |  Negative: {1 - pos_share:.1%}")
print(f"Majority-class baseline accuracy: {max(pos_share, 1 - pos_share):.1%}")

## 5. Tokenize and build Word2Vec embeddings

In [ ]:
MAX_LEN = 200
VOCAB_SIZE = 1000
EMBEDDING_DIM = 128

tokenizer = Tokenizer(num_words=VOCAB_SIZE)
tokenizer.fit_on_texts(df["clean"])
word_index = tokenizer.word_index
sequences = tokenizer.texts_to_sequences(df["clean"])
X = pad_sequences(sequences, padding="post", maxlen=MAX_LEN)

w2v = Word2Vec(sentences=[s.split() for s in df["clean"]],
               vector_size=EMBEDDING_DIM, window=5, min_count=1, workers=4)

embedding_matrix = np.zeros((len(word_index) + 1, EMBEDDING_DIM))
for word, i in word_index.items():
    if word in w2v.wv:
        embedding_matrix[i] = w2v.wv[word]

## 6. Model: Word2Vec (frozen) → Bidirectional LSTM → Dense

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, labels, test_size=0.2, random_state=42)

model = Sequential([
    Embedding(input_dim=len(word_index) + 1, output_dim=EMBEDDING_DIM,
              weights=[embedding_matrix], input_length=MAX_LEN, trainable=False),
    Bidirectional(LSTM(64)),
    Dropout(0.5),
    Dense(64, activation="relu"),
    Dense(32, activation="relu"),
    Dense(1, activation="sigmoid"),
])
model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
model.summary()

history = model.fit(X_train, y_train, batch_size=32, epochs=10,
                    validation_data=(X_test, y_test))

## 7. Evaluation

In [ ]:
probs = model.predict(X_test).ravel()
preds = (probs >= 0.5).astype(int)

print(f"Accuracy : {accuracy_score(y_test, preds):.3f}")
print(f"F1 score : {f1_score(y_test, preds):.3f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, probs):.3f}")
print(classification_report(y_test, preds, target_names=["negative", "positive"]))

sns.heatmap(confusion_matrix(y_test, preds), annot=True, fmt="d", cmap="Blues",
            xticklabels=["negative", "positive"], yticklabels=["negative", "positive"])
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion matrix")
plt.show()

plt.plot(history.history["accuracy"], label="train")
plt.plot(history.history["val_accuracy"], label="validation")
plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.legend(); plt.title("Training curve")
plt.show()

## 8. Save model and tokenizer for the Streamlit app

In [ ]:
model.save("sentiment_model.h5")
joblib.dump(tokenizer, "tokenizer.pkl")

## 9. Try it

In [ ]:
def predict_sentiment(review: str) -> float:
    seq = pad_sequences(tokenizer.texts_to_sequences([preprocess(review)]),
                        padding="post", maxlen=MAX_LEN)
    return float(model.predict(seq, verbose=0)[0][0])

for review in ["bad services", "awesome product, worth every penny", "not good, stopped working"]:
    p = predict_sentiment(review)
    print(f"{review!r:45} -> {'positive' if p >= 0.5 else 'negative'} ({p:.2f})")